<a href="https://colab.research.google.com/github/ipreencekmr/anthropic/blob/main/Using_MCP_Server.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%pip install anthropic

In [ ]:
%pip install "anthropic[mcp]"

### Setup

In [7]:
#Load Environment Variable
import os
from google.colab import userdata

# Retrieve from Colab Secrets and set to environment variables
os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
os.environ["GITHUB_TOKEN"] = userdata.get("PAT")

In [8]:
import os
import anthropic

client = anthropic.Anthropic()
github_token = os.environ["GITHUB_TOKEN"]

### Using GitHub MCP Server

In [9]:
mcp_server_config = {
    "type": "url",
    "url": "https://api.githubcopilot.com/mcp/",
    "name": "github",
    "authorization_token": github_token
    }

tools_config = {
    "type": "mcp_toolset",
    "mcp_server_name": "github",
    "configs": {
        "delete_repository": {
             "enabled": False
             }
        }
    }

In [10]:
def chat(query):
  response = client.beta.messages.create(
    model="claude-sonnet-4-6",
    max_tokens=1024,
    messages=[
        {
            "role": "user",
            "content": query
        }
    ],
    mcp_servers=[
        mcp_server_config
    ],
    tools=[
        tools_config
    ],
    betas=["mcp-client-2025-11-20"],
  )

  return response

### Execution

In [ ]:
prompt = "List the 5 most recently worked repository list"

result = chat(prompt)

result


### Pretty Print

In [14]:
for block in result.content:
    if block.type == "text":
        print(block.text)

Here are your **5 most recently worked on repositories**:

| # | Repository | Language | Description | Last Updated |
|---|-----------|----------|-------------|--------------|
| 1 | [anthropic](https://github.com/ipreencekmr/anthropic) | Jupyter Notebook | Collab Notebook Exercise | Jun 28, 2026 |
| 2 | [agentic-chess](https://github.com/ipreencekmr/agentic-chess) | Python | _(no description)_ | Jun 25, 2026 |
| 3 | [shopping-cart](https://github.com/ipreencekmr/shopping-cart) | JavaScript | A React-based shopping cart app with Unit Testing, E2E (Playwright), and A11y (axe-core) testing | Jun 2, 2026 |
| 4 | [rag-platform](https://github.com/ipreencekmr/rag-platform) | Python | _(no description)_ | May 1, 2026 |
| 5 | [resume-tailor](https://github.com/ipreencekmr/resume-tailor) | Python | _(no description)_ | Apr 21, 2026 |

These are sorted by the most recently updated. Would you like more details about any of these repositories?


In [ ]:
import json
for block in result.content:
    if block.type == "text":
        print("📝 Claude's answer:\n")
        print(block.text)
        print()

    elif block.type == "mcp_tool_use":
        print(f"🔧 Called tool: {block.name}  (server: {block.server_name})")
        print("   Input:")
        print(json.dumps(block.input, indent=2))
        print()

    elif block.type == "mcp_tool_result":
        status = "❌ ERROR" if block.is_error else "✅ SUCCESS"
        print(f"📦 Tool result ({status}):")
        for item in block.content:
            if item.type == "text":
                # Try to pretty-print as JSON if the tool returned JSON text,
                # otherwise just print it as-is
                try:
                    parsed = json.loads(item.text)
                    print(json.dumps(parsed, indent=2))
                except json.JSONDecodeError:
                    print(item.text)
        print()